In [1]:
# ============================================================
# NOTEBOOK 03 — VECTORIZATION (EMBEDDINGS)
# ============================================================
# WHAT IS VECTORIZATION?
# Each text chunk needs to be converted into a list of numbers
# called an "embedding" or "vector".
#
# These numbers capture the MEANING of the text.
# Similar text → similar numbers → close together in space
# Different text → different numbers → far apart in space
#
# We use a FREE model from HuggingFace:
# "sentence-transformers/all-MiniLM-L6-v2"
# This model runs LOCALLY on your machine — no API key needed!
# It converts any text into a vector of 384 numbers.
#
# After vectorization we store everything in ChromaDB
# which is a vector database — like a normal database
# but designed to store and search vectors efficiently.
# ============================================================

print("Starting vectorization notebook...")

Starting vectorization notebook...


In [2]:
# Load environment variables from our .env file
# This is how we safely use API keys without hardcoding them
from dotenv import load_dotenv
load_dotenv()  # reads .env file and loads variables into environment

# PyPDFLoader to reload our document
from langchain_community.document_loaders import PyPDFLoader

# Text splitter from notebook 02
from langchain.text_splitter import RecursiveCharacterTextSplitter

# HuggingFaceEmbeddings loads a local embedding model
# It runs on your machine — completely free, no API needed
from langchain_huggingface import HuggingFaceEmbeddings

# Chroma is our vector database
# It stores embeddings on disk so we don't recalculate every time
from langchain_community.vectorstores import Chroma

import os

print("Libraries imported!")

Libraries imported!


In [3]:
# ============================================================
# STEP 1: RELOAD DOCUMENTS AND CREATE CHUNKS
# ============================================================
# We repeat ingestion and chunking here so this notebook
# is self-contained and can run independently

# Load the PDF
pdf_path = "../data/documents/attention_is_all_you_need.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Split into chunks — same settings as notebook 02
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,    # 500 characters per chunk
    chunk_overlap=50   # 50 character overlap between chunks
)
chunks = text_splitter.split_documents(documents)

print(f"Pages loaded  : {len(documents)}")
print(f"Chunks created: {len(chunks)}")

Pages loaded  : 15
Chunks created: 94


In [4]:
# ============================================================
# STEP 2: LOAD THE EMBEDDING MODEL
# ============================================================

# We use sentence-transformers/all-MiniLM-L6-v2
# This is a small but powerful model that:
# - Runs completely locally on your machine
# - Converts any text into 384 numbers
# - Is free and fast
# - Is one of the most popular embedding models in the industry

print("Loading embedding model... (first time takes 1-2 minutes to download)")

embedding_model = HuggingFaceEmbeddings(

    # The model name from HuggingFace
    model_name="sentence-transformers/all-MiniLM-L6-v2",

    # Run on CPU — change to "cuda" if you have a GPU
    model_kwargs={"device": "cpu"},

    # Normalise embeddings so distances are consistent
    encode_kwargs={"normalize_embeddings": True}
)

print("Embedding model loaded!")

Loading embedding model... (first time takes 1-2 minutes to download)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


In [5]:
# ============================================================
# STEP 3: SEE WHAT AN EMBEDDING ACTUALLY LOOKS LIKE
# ============================================================

# Let's embed a simple sentence to understand what we get back
sample_text = "What is the attention mechanism in transformers?"

# embed_query converts one piece of text into a vector
sample_embedding = embedding_model.embed_query(sample_text)

print(f"Text   : {sample_text}")
print(f"Type   : {type(sample_embedding)}")
print(f"Length : {len(sample_embedding)} numbers")  # Should be 384
print(f"First 10 numbers: {sample_embedding[:10]}")

# These numbers might look random but they encode
# the full meaning of the sentence mathematically

Text   : What is the attention mechanism in transformers?
Type   : <class 'list'>
Length : 384 numbers
First 10 numbers: [-0.015864567831158638, 0.013766015879809856, -0.05919903516769409, -0.011535711586475372, 0.011662348173558712, 0.0362069234251976, 0.04548201709985733, 0.06301344186067581, 0.07075751572847366, 0.03877071291208267]


In [6]:
# ============================================================
# STEP 4: PROVE THAT SIMILAR TEXT = SIMILAR VECTORS
# ============================================================
# This is the core concept behind RAG!

import numpy as np

# Three sentences — two similar, one different
sentence_a = "The attention mechanism helps the model focus"
sentence_b = "Attention allows the network to concentrate on relevant parts"
sentence_c = "I love eating pizza on Fridays"

# Get embeddings for all three
emb_a = embedding_model.embed_query(sentence_a)
emb_b = embedding_model.embed_query(sentence_b)
emb_c = embedding_model.embed_query(sentence_c)

# Calculate cosine similarity between them
# Cosine similarity: 1.0 = identical meaning, 0.0 = no relation
def cosine_similarity(v1, v2):
    # Dot product divided by product of magnitudes
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sim_ab = cosine_similarity(emb_a, emb_b)
sim_ac = cosine_similarity(emb_a, emb_c)

print(f"Sentence A : {sentence_a}")
print(f"Sentence B : {sentence_b}")
print(f"Sentence C : {sentence_c}")
print(f"\nSimilarity A vs B (similar meaning) : {sim_ab:.4f}")
print(f"Similarity A vs C (different meaning): {sim_ac:.4f}")
print("\nHigher number = more similar meaning!")

Sentence A : The attention mechanism helps the model focus
Sentence B : Attention allows the network to concentrate on relevant parts
Sentence C : I love eating pizza on Fridays

Similarity A vs B (similar meaning) : 0.7127
Similarity A vs C (different meaning): -0.0317

Higher number = more similar meaning!


In [7]:
# ============================================================
# STEP 5: STORE ALL CHUNKS IN CHROMADB
# ============================================================
# ChromaDB is our vector database.
# It takes all our chunks, embeds them, and stores them on disk.
# Later we can search it with a question and get
# the most relevant chunks back instantly.

# Where to save the ChromaDB database on disk
chroma_path = "../.chroma"

print("Creating ChromaDB vector store...")
print("This embeds all chunks — takes 1-2 minutes...")

# Chroma.from_documents does three things at once:
# 1. Takes all our text chunks
# 2. Runs each one through the embedding model
# 3. Stores the text + embedding in the database on disk
vectorstore = Chroma.from_documents(

    # All our text chunks from the PDF
    documents=chunks,

    # The embedding model to convert text to numbers
    embedding=embedding_model,

    # Where to save the database on disk
    persist_directory=chroma_path
)

print(f"Vector store created!")
print(f"Total vectors stored: {vectorstore._collection.count()}")

Creating ChromaDB vector store...
This embeds all chunks — takes 1-2 minutes...
Vector store created!
Total vectors stored: 94


In [8]:
# ============================================================
# STEP 6: TEST THE VECTOR STORE WITH A QUESTION
# ============================================================
# Let's prove it works by asking a question and seeing
# which chunks it finds as most relevant

# Our test question
question = "How does the attention mechanism work?"

# similarity_search finds the most relevant chunks
# k=3 means return the top 3 most similar chunks
results = vectorstore.similarity_search(question, k=3)

print(f"Question: {question}")
print(f"\nTop 3 most relevant chunks found:\n")

for i, result in enumerate(results):
    print(f"--- RESULT {i+1} ---")
    # Show which page this chunk came from
    print(f"Source page : {result.metadata.get('page', 'unknown')}")
    # Show the chunk text
    print(f"Content     : {result.page_content[:300]}")
    print()

Question: How does the attention mechanism work?

Top 3 most relevant chunks found:

--- RESULT 1 ---
Source page : 12
Content     : .
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more diffic

--- RESULT 2 ---
Source page : 14
Content     : but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the
sentence. We give two such examples above, from two different heads from the encoder self-attention
at lay

--- RESULT 3 ---
Source page : 6
Content     : from our models and present and discuss examples in the appendix. Not only do individual attention
heads clearly learn to perform different tasks, many appear to exhibi

In [9]:
# ============================================================
# SUMMARY — WHAT DID WE LEARN?
# ============================================================

print("=== VECTORIZATION SUMMARY ===")
print(f"Embedding model : sentence-transformers/all-MiniLM-L6-v2")
print(f"Vector size     : 384 numbers per chunk")
print(f"Total vectors   : {vectorstore._collection.count()}")
print(f"Database saved  : {chroma_path}")

print("""
WHAT HAPPENS NEXT?
------------------
We now have a vector database full of embedded chunks.

The next step is RETRIEVAL — given a user question,
we search the vector database and pull out the most
relevant chunks to use as context for the LLM.

See notebook 04_retrieval.ipynb for the next step!
""")

=== VECTORIZATION SUMMARY ===
Embedding model : sentence-transformers/all-MiniLM-L6-v2
Vector size     : 384 numbers per chunk
Total vectors   : 94
Database saved  : ../.chroma

WHAT HAPPENS NEXT?
------------------
We now have a vector database full of embedded chunks.

The next step is RETRIEVAL — given a user question,
we search the vector database and pull out the most
relevant chunks to use as context for the LLM.

See notebook 04_retrieval.ipynb for the next step!

